# DeltaWatch: fresh-session demo (Colab, CPU)
Clones the public repo, runs the core from `config/default.yaml` + the frozen `examples/manifest.json`, then runs the tests
and bundles everything into one zip. All logic lives in `src/deltawatch/`; this notebook only calls it.

Contains modified Copernicus Sentinel data 2025, accessed via Element 84 Earth Search.

In [ ]:
# 1. Get the repository into /content/deltawatch (fresh clone of the public repo)
import os
from pathlib import Path
if not Path("/content/deltawatch/pyproject.toml").exists():
    !git clone --depth 1 https://github.com/ayatfakhry/deltawatch.git /content/deltawatch
os.chdir("/content/deltawatch")
print(sorted(p.name for p in Path(".").iterdir()))

In [ ]:
# 2. Install dependencies (Colab already has numpy, pandas, matplotlib, PyYAML)
!pip -q install "rasterio>=1.4" "pystac-client>=0.8" pytest pytest-cov ruff==0.16.9
import sys
sys.path.insert(0, "src")          # no kernel restart needed (an editable install would need one)
import deltawatch; print("deltawatch", deltawatch.__version__)

In [ ]:
# 3. Offline tests (synthetic rasters, no network) + lint
!python -m pytest --cov=deltawatch --cov-report=term 2>&1 | tee outputs_tests.txt | tail -20
!ruff check . && ruff format --check .

In [ ]:
# 4. Run the core from config + frozen manifest
import logging, json
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
from deltawatch.config import load_config
from deltawatch.pipeline import run_core
cfg = load_config("config/default.yaml")
res = run_core(cfg, "examples/manifest.json", "outputs/run")
qa, meta = res["qa"], res["meta"]
print(qa[["date", "platform", "relative_orbit", "valid_fraction", "valid_fraction_if_dilated", "ndvi_p50",
          "ndvi_out_of_range_px", "ndvi_out_of_range_with_positive_refl_px", "roundtrip_ok"]].to_string(index=False))
print(json.dumps({"timings": meta["timings_s_measured"], "memory": meta["memory"],
                  "local_MB": meta["measured_local_outputs_MB"]}, indent=1))

In [ ]:
# 5. QA panel
from IPython.display import Image, display
display(Image("outputs/run/qa_ndvi_panel.png", width=1100))

In [ ]:
# 6. Acceptance checks for a fresh-session run
import pandas as pd
ref = pd.read_csv("examples/qa_per_date.csv")
cols = ["date", "valid_fraction", "ndvi_p2", "ndvi_p50", "ndvi_p98", "nonpositive_reflectance_px"]
checks = {
    "all 8 dates processed": len(qa) == 8,
    "all GeoTIFFs round-trip": bool(qa.roundtrip_ok.all()),
    "no out-of-range NDVI with positive reflectances": int(qa.ndvi_out_of_range_with_positive_refl_px.sum()) == 0,
    "exported valid NDVI within [-1, 1]": bool((qa.valid_ndvi_max_abs <= 1).all()),
    "source nodata never valid": int(qa.nodata_marked_valid_px.sum()) == 0,
    "QA identical to the frozen reference run": bool((qa[cols].round(6).values == ref[cols].round(6).values).all()),
    "peak RSS < 2 GB": meta["memory"]["measured_peak_process_rss_MB"] < 2000,
    "local outputs < 500 MB": meta["measured_local_outputs_MB"] < 500,
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
print(f"{sum(checks.values())}/{len(checks)} checks passed")

In [ ]:
# 7. Optional live smoke test (anonymous read of one real COG window)
!DELTAWATCH_LIVE=1 python -m pytest tests/live -q 2>&1 | tail -3

In [ ]:
# 7b. Optional locked benchmark (validation first, then the locked test; nothing is tuned)
from deltawatch.benchmark import run_benchmark
from deltawatch.manifest import load_manifest
bench = run_benchmark(cfg, load_manifest("examples/manifest.json"), "outputs/run", "outputs/run/benchmark")
for split in ("validation", "test"):
    s = bench[split]["scores"]
    print(split, bench[split]["target_date"], {m: (s[m]["n_predicted"], s[m]["coverage"], s[m]["mae"], s[m]["rmse"]) for m in ("latest", "median3")})
ref = json.load(open("examples/benchmark_results.json"))
checks["benchmark identical to reference run"] = all(
    bench[sp]["scores"][m] == ref[sp]["scores"][m] for sp in ("validation", "test") for m in ("latest", "median3"))
print("benchmark identical to reference run:", checks["benchmark identical to reference run"])
display(Image("outputs/run/benchmark/benchmark_test_panel.png", width=1100))

In [ ]:
# 7c. Results-preservation check against the frozen v0.2 reference outputs (offline)
!PYTHONPATH=src python -m deltawatch.pipeline compare --run-dir outputs/run --report outputs/run/compare_report.json

In [ ]:
# 8. Bundle everything into ONE zip to send back (run this even if an earlier cell failed)
import json, sys, traceback, zipfile
from pathlib import Path
report = {"checks": globals().get("checks"),
          "last_error": "".join(traceback.format_exception(sys.last_type, sys.last_value, sys.last_traceback))
                        if hasattr(sys, "last_value") else None}
Path("outputs").mkdir(exist_ok=True)
Path("outputs/report.json").write_text(json.dumps(report, indent=2, default=str))
bundle = Path("deltawatch_results_colab.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in list(Path("outputs").rglob("*")) + [Path("outputs_tests.txt")]:
        if f.is_file() and f.suffix.lower() not in {".tif", ".tiff"}:
            z.write(f)
print(bundle, round(bundle.stat().st_size / 1e6, 2), "MB")
try:
    from google.colab import files; files.download(str(bundle))
except ImportError:
    print("Not in Colab: download the zip manually.")